# Exploración visual del dataset BreastDCEDL

Objetivo: **entender los datos antes de construir la CNN**. Este notebook es solo para
*mirar*: las comprobaciones formales están en `src/eda.py`.

Regla: todo lo que compara **pCR=0 frente a pCR=1** se hace **solo con train**. De test
solo contamos pacientes, para no mirar el conjunto de evaluación final antes de tiempo.

Funciona en **Google Colab** (descarga los datos automáticamente) o en local si el
repositorio tiene `data/breastdcedl/`.

## 0. Preparación: datos y librerías

In [ ]:
import os, sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
URL_SCRIPT = ("https://storage.googleapis.com/usecasesf-breastdcedl-alumnos-8264/"
              "breastdcedl/descargar_datos.py")

if os.environ.get("BDCEDL_RAIZ"):                  # ruta forzada a mano
    RAIZ = Path(os.environ["BDCEDL_RAIZ"])
elif EN_COLAB:
    RAIZ = Path("/content/breastdcedl")
    if not (RAIZ / "metadata" / "samples.csv").exists():
        # Descarga del bucket público (tarda unos minutos la primera vez)
        !curl -sL -o /content/descargar_datos.py {URL_SCRIPT}
        !python /content/descargar_datos.py --destino {RAIZ} --hilos 32
else:
    # En local: busca data/breastdcedl subiendo desde la carpeta del notebook
    RAIZ = next((p / "data" / "breastdcedl" for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "data" / "breastdcedl" / "metadata" / "samples.csv").exists()), None)
    assert RAIZ is not None, "No encuentro data/breastdcedl. Ejecuta el notebook dentro del repo."

sys.path.insert(0, str(RAIZ))
import utils_caso as uc                             # utilidades del profesor
print("Datos en:", RAIZ)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})

COLOR = {0: "#2a78d6", 1: "#eb6834"}               # pCR=0 azul, pCR=1 naranja
ETQ = {0: "no pCR (0)", 1: "pCR (1)"}
FASES = ("PRE", "EARLY", "LATE")
rng = np.random.default_rng(42)                     # semilla fija: mismas figuras siempre

samples = uc.cargar_samples(RAIZ)                   # 1 fila por corte
patients = uc.cargar_patients(RAIZ)                 # 1 fila por paciente

# Tabla por paciente con la cohorte y el subtipo, que usaremos en todo el notebook
pac = (samples.groupby("patient_id")
       .agg(split=("split", "first"), pCR=("pCR", "first"), fold=("fold", "first"),
            n_cortes=("sample_id", "size"))
       .join(patients.set_index("pid")[["dataset", "HR_HER2_STATUS", "age", "tum_vol"]]))
train_pac = pac[pac.split == "train"]
print(f"{len(samples)} cortes, {len(pac)} pacientes")

## 1. Qué hay en los CSV

- `samples.csv`: **una fila por corte**, con las rutas de sus 3 PNG (PRE, EARLY, LATE),
  la etiqueta y el fold.
- `patients.csv`: **una fila por paciente**, con variables clínicas. Solo sirven para
  interpretar; la red **no** las recibe.

In [ ]:
samples.head()

In [ ]:
patients.head()

## 2. Cuántos datos hay y cómo están las clases

La unidad estadística es la **paciente**. Los ~10 cortes de una paciente son rebanadas
vecinas de la misma mama: son muy parecidos entre sí, no son 10 casos independientes.

In [ ]:
print("Pacientes por split y clase:")
display(pac.groupby(["split", "pCR"]).size().unstack())
print("Cortes por split y clase:")
display(samples.groupby(["split", "pCR"]).size().unstack())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
t = pac.groupby(["split", "pCR"]).size().unstack().reindex(["train", "test"])
x = np.arange(2)
for k, c in enumerate((0, 1)):
    b = axes[0].bar(x + (k - .5) * .38, t[c], .36, color=COLOR[c], label=ETQ[c])
    axes[0].bar_label(b, padding=2, fontsize=9)
axes[0].set_xticks(x, t.index); axes[0].set_ylabel("pacientes")
axes[0].set_title("Pacientes por split y clase"); axes[0].legend(frameon=False)

cuentas = pac.n_cortes.value_counts().sort_index()
axes[1].bar(cuentas.index, cuentas.values, color=COLOR[0])
axes[1].set_xticks(cuentas.index); axes[1].set_xlabel("cortes por paciente")
axes[1].set_title("Cortes por paciente")
plt.tight_layout(); plt.show()

n0, n1 = (samples[samples.split == "train"].pCR.value_counts().sort_index())
print(f"pos_weight = N0/N1 = {n0}/{n1} = {n0/n1:.2f}")
print(f"Predecir siempre 0 daría accuracy = {1 - pac[pac.split=='test'].pCR.mean():.1%} en test")

## 3. Separación por paciente y folds

Ninguna paciente puede estar en dos splits ni en dos folds; si no, habría **fuga de datos**.
Los 5 folds dividen train para la validación interna.

In [ ]:
print("Pacientes en más de un split:", (samples.groupby("patient_id").split.nunique() > 1).sum())
print("Pacientes de train en más de un fold:",
      (samples[samples.split == "train"].groupby("patient_id").fold.nunique() > 1).sum())

folds = train_pac.groupby("fold").agg(pacientes=("pCR", "size"), prop_pCR=("pCR", "mean"))
display(folds.round(3))
ax = folds.prop_pCR.plot.bar(color=COLOR[1], figsize=(5, 3), rot=0)
ax.axhline(train_pac.pCR.mean(), color="gray", ls="--", lw=1)
ax.set_ylabel("proporción pCR"); ax.set_title("pCR por fold (línea = media de train)")
plt.tight_layout(); plt.show()

## 4. Ver un corte: las tres fases y el realce

- **PRE**: antes del contraste. **EARLY**: poco después. **LATE**: varios minutos después.
- **Realce = EARLY − PRE**: lo que se ilumina al llegar el contraste. Es la señal del problema.

Cambia `SAMPLE_ID` por cualquier corte de `samples.csv`.

In [ ]:
def cargar(sample_id):
    fila = samples.loc[samples.sample_id == sample_id].iloc[0]
    return fila, uc.cargar_imagen(fila, RAIZ)       # (3, 256, 256) en [0, 1]: lo que verá la red

def ver_corte(sample_id):
    fila, x = cargar(sample_id)
    realce = x[1] - x[0]
    fig, axes = plt.subplots(1, 5, figsize=(17, 3.8))
    for ax, img, fase in zip(axes, x, FASES):
        ax.imshow(img, cmap="gray", vmin=0, vmax=1); ax.set_title(fase)
    axes[3].imshow(np.clip(realce, 0, None), cmap="magma", vmin=0, vmax=0.4)
    axes[3].set_title("REALCE (EARLY - PRE)")
    axes[4].imshow(x[2] - x[1], cmap="coolwarm", vmin=-0.2, vmax=0.2)
    axes[4].set_title("LATE - EARLY  (azul = lavado)")
    for ax in axes: ax.axis("off")
    fig.suptitle(f"{sample_id} | paciente {fila.patient_id} | {fila.split} | pCR={fila.pCR}")
    plt.tight_layout(); plt.show()
    print("forma:", x.shape, "| rango:", round(float(x.min()), 3), "-", round(float(x.max()), 3))

SAMPLE_ID = samples[samples.split == "train"].sample_id.iloc[0]
ver_corte(SAMPLE_ID)

## 5. Ver todos los cortes de una paciente

Los cortes `z` son **alturas** distintas de la mama (no instantes de tiempo).

In [ ]:
def ver_paciente(patient_id, modo="EARLY"):
    cortes = samples[samples.patient_id == patient_id].sort_values("slice_index")
    n = len(cortes)
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 2.6))
    for ax, fila in zip(np.atleast_1d(axes), cortes.itertuples()):
        x = uc.cargar_imagen(fila, RAIZ)
        if modo == "REALCE":
            ax.imshow(np.clip(x[1] - x[0], 0, None), cmap="magma", vmin=0, vmax=0.4)
        else:
            ax.imshow(x[FASES.index(modo)], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"z={fila.slice_index}", fontsize=9); ax.axis("off")
    info = pac.loc[patient_id]
    fig.suptitle(f"{patient_id} | {info.dataset} | {info.HR_HER2_STATUS} | pCR={info.pCR} | {modo}")
    plt.tight_layout(); plt.show()

PACIENTE = samples.loc[samples.sample_id == SAMPLE_ID, "patient_id"].iloc[0]
ver_paciente(PACIENTE, "EARLY")
ver_paciente(PACIENTE, "REALCE")

## 6. pCR=0 frente a pCR=1 (train)

Corte central de 3 pacientes de cada clase, elegidas al azar. **Pregunta:** ¿ves a simple
vista algo que distinga las dos clases? Si no, es normal: el problema es difícil.

In [ ]:
def comparar_clases(n=3):
    tr = train_pac
    filas = [(pid, c) for c in (0, 1)
             for pid in rng.choice(tr[tr.pCR == c].index, n, replace=False)]
    fig, axes = plt.subplots(len(filas), 4, figsize=(10, 2.6 * len(filas)))
    for fila_ax, (pid, c) in zip(axes, filas):
        cortes = samples[samples.patient_id == pid].sort_values("slice_index")
        fila = cortes.iloc[len(cortes) // 2]
        x = uc.cargar_imagen(fila, RAIZ)
        for ax, img, fase in zip(fila_ax[:3], x, FASES):
            ax.imshow(img, cmap="gray", vmin=0, vmax=1); ax.set_title(fase, fontsize=9)
        fila_ax[3].imshow(np.clip(x[1] - x[0], 0, None), cmap="magma", vmin=0, vmax=0.4)
        fila_ax[3].set_title("realce", fontsize=9)
        for ax in fila_ax: ax.set_xticks([]); ax.set_yticks([])
        fila_ax[0].set_ylabel(f"{pid}\n{pac.loc[pid, 'dataset']}\n{ETQ[c]}",
                              fontsize=8, color=COLOR[c])
    plt.tight_layout(); plt.show()

comparar_clases(3)

## 7. Cohortes: ¿se distinguen las imágenes de Duke, I-SPY1 e I-SPY2?

Si cada cohorte "se ve distinta" (escáner, protocolo, recorte) **y** tiene distinta tasa de
pCR, la red podría aprender a reconocer la cohorte en vez de la biología del tumor.

In [ ]:
tasa = train_pac.groupby("dataset").pCR.agg(pacientes="size", tasa_pCR="mean")
display(tasa.round(3))

cohortes = sorted(train_pac.dataset.dropna().unique())
fig, axes = plt.subplots(len(cohortes), 4, figsize=(10, 2.7 * len(cohortes)))
for fila_ax, coh in zip(np.atleast_2d(axes), cohortes):
    pid = rng.choice(train_pac[train_pac.dataset == coh].index)
    cortes = samples[samples.patient_id == pid].sort_values("slice_index")
    x = uc.cargar_imagen(cortes.iloc[len(cortes) // 2], RAIZ)
    for ax, img, fase in zip(fila_ax[:3], x, FASES):
        ax.imshow(img, cmap="gray", vmin=0, vmax=1); ax.set_title(fase, fontsize=9)
    fila_ax[3].imshow(np.clip(x[1] - x[0], 0, None), cmap="magma", vmin=0, vmax=0.4)
    fila_ax[3].set_title("realce", fontsize=9)
    for ax in fila_ax: ax.set_xticks([]); ax.set_yticks([])
    fila_ax[0].set_ylabel(f"{coh}\n{pid}", fontsize=9)
plt.tight_layout(); plt.show()

## 8. Variables clínicas: subtipo tumoral, edad y volumen (train)

No entran en la red, pero ayudan a interpretar: en la literatura, los tumores **HER2+** y
**triple negativo** alcanzan pCR con más frecuencia que los **HR+/HER2−**.

In [ ]:
print("% de valores faltantes:")
display((patients[["HR_HER2_STATUS", "age", "tum_vol", "menopause"]].isna().mean() * 100).round(1))

sub = train_pac.groupby("HR_HER2_STATUS").pCR.agg(pacientes="size", tasa_pCR="mean")
display(sub.round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
sub.tasa_pCR.sort_values().plot.barh(ax=axes[0], color=COLOR[1])
axes[0].axvline(train_pac.pCR.mean(), color="gray", ls="--", lw=1)
axes[0].set_title("Tasa de pCR por subtipo"); axes[0].set_ylabel("")
for c in (0, 1):
    d = train_pac[train_pac.pCR == c]
    axes[1].hist(d.age.dropna(), bins=20, alpha=.6, color=COLOR[c], label=ETQ[c], density=True)
    axes[2].hist(np.log10(d.tum_vol.dropna()), bins=25, alpha=.6, color=COLOR[c],
                 label=ETQ[c], density=True)
axes[1].set_title("Edad"); axes[1].legend(frameon=False)
axes[2].set_title("Volumen tumoral (log10)"); axes[2].legend(frameon=False)
plt.tight_layout(); plt.show()

## 9. Intensidades y curva de realce (train)

Medimos la intensidad media **en el tejido** (PRE > 0.1, para quitar el fondo negro) en cada
fase. Para que sea rápido usamos una muestra de pacientes; sube `N_POR_CLASE` si quieres más.

- Si **PRE < EARLY** en todas, las fases están bien ordenadas.
- La **curva PRE → EARLY → LATE** resume la cinética del contraste; si baja al final es **lavado**.

In [ ]:
N_POR_CLASE = 150            # pacientes de train por clase (None = todas; tarda más)

def medir(fila):
    x = uc.cargar_imagen(fila, RAIZ)
    tejido = x[0] > 0.1
    return [float(c[tejido].mean()) if tejido.any() else np.nan for c in x]

elegidas = []
for c in (0, 1):
    ids = train_pac[train_pac.pCR == c].index
    elegidas += list(ids if N_POR_CLASE is None else rng.choice(ids, min(N_POR_CLASE, len(ids)), replace=False))
sub_s = samples[samples.patient_id.isin(elegidas)]
med = pd.DataFrame([medir(f) for f in sub_s.itertuples()], columns=list(FASES), index=sub_s.index)
med = sub_s[["patient_id"]].join(med).groupby("patient_id").mean().join(pac[["pCR", "dataset"]])
med["realce"] = med.EARLY - med.PRE
med["lavado"] = med.LATE < med.EARLY
print(f"{len(med)} pacientes medidas")
print(f"PRE < EARLY en el {(med.PRE < med.EARLY).mean():.1%} | lavado en el {med.lavado.mean():.1%}")
display(med.groupby("pCR")[["PRE", "EARLY", "LATE", "realce", "lavado"]].mean().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

# a) PRE frente a EARLY: todos los puntos deben quedar por encima de la diagonal
for c in (0, 1):
    d = med[med.pCR == c]
    axes[0].scatter(d.PRE, d.EARLY, s=10, alpha=.6, color=COLOR[c], label=ETQ[c])
lim = [med[["PRE", "EARLY"]].min().min() * .95, med[["PRE", "EARLY"]].max().max() * 1.05]
axes[0].plot(lim, lim, "--", color="gray", lw=1)
axes[0].set_xlabel("PRE"); axes[0].set_ylabel("EARLY")
axes[0].set_title("Orden de fases"); axes[0].legend(frameon=False)

# b) Curva media de realce por clase
for c in (0, 1):
    d = med[med.pCR == c][list(FASES)]
    m, e = d.mean(), d.std() / np.sqrt(len(d))
    axes[1].errorbar(range(3), m, yerr=e, marker="o", capsize=3, color=COLOR[c], label=ETQ[c])
axes[1].set_xticks(range(3), FASES); axes[1].set_ylabel("intensidad media en tejido")
axes[1].set_title("Curva de realce media (± error estándar)"); axes[1].legend(frameon=False)

# c) Realce por cohorte y clase
cohortes = sorted(med.dataset.dropna().unique())
for k, c in enumerate((0, 1)):
    datos = [med[(med.dataset == coh) & (med.pCR == c)].realce.dropna() for coh in cohortes]
    bp = axes[2].boxplot(datos, positions=np.arange(len(cohortes)) + (k - .5) * .36, widths=.3,
                         patch_artist=True, showfliers=False)
    for b in bp["boxes"]: b.set(facecolor=COLOR[c], edgecolor=COLOR[c], alpha=.8)
axes[2].set_xticks(range(len(cohortes)), cohortes)
axes[2].set_ylabel("realce EARLY - PRE"); axes[2].set_title("Realce por cohorte y clase")
plt.tight_layout(); plt.show()

## 10. Distribución de intensidades por cohorte

Histograma de los píxeles de tejido de EARLY. Si las cohortes tienen distribuciones muy
distintas, es una diferencia técnica (no biológica) que la red podría aprovechar.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
for coh in sorted(train_pac.dataset.dropna().unique()):
    ids = rng.choice(train_pac[train_pac.dataset == coh].index, 20, replace=False)
    valores = []
    for fila in samples[samples.patient_id.isin(ids)].itertuples():
        x = uc.cargar_imagen(fila, RAIZ)
        valores.append(x[1][x[0] > 0.1])
    ax.hist(np.concatenate(valores), bins=60, histtype="step", density=True, lw=1.8, label=coh)
ax.set_xlabel("intensidad EARLY en tejido"); ax.set_ylabel("densidad")
ax.set_title("Intensidades por cohorte (20 pacientes de cada una)"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 11. Conclusiones (rellénalas tú)

Anota lo que observes; esto va a la **diapositiva 2** y a la sección de datos del informe.

1. **Tamaño y desbalance:** …
2. **Separación por paciente y folds:** …
3. **¿Se distinguen pCR=0 y pCR=1 a simple vista o en el realce medio?** …
4. **¿Hay diferencias entre cohortes (tasa de pCR, aspecto, intensidades)? ¿Qué riesgo supone?** …
5. **Subtipo tumoral: ¿qué implica para interpretar los resultados del modelo?** …
6. **Decisiones para la CNN que salen de aquí** (pos_weight, métricas, aumento de datos solo geométrico…): …